# 05. Simulasi Operasional & Disagregasi Bisnis (Manajemen Persediaan & Service Level 95%)

**Naskah Akademik & Pembuktian Teoretis Skripsi**  
*Program Studi Sistem Informasi / Ilmu Komputer*

Notebook ini mendemonstrasikan fase terakhir dari arsitektur *forecasting*, yaitu **Simulasi Operasional 12-Minggu** dan **Disagregasi Bisnis**. 
Berdasarkan audit forensik pada sistem sebelumnya (yang menghasilkan *Food Waste*), fungsi agregasi ditata ulang menjadi:
1. **Adaptive Day-of-Week (DoW) Share** (Jendela 4-Minggu).
2. **Rolling 12-Week SKU Share** (Bukan *all-time* historis).
3. **MTO (Make-to-Order) Threshold**: Varian dengan target $< 0.5$ akan dipaksa menjadi 0 (dialihkan ke *pre-order*).


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
import os
import json

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
os.makedirs('Visualisasi_Gambar', exist_ok=True)

# Load Data
df_weekly = pd.read_csv('../01_data_ITA-Cake-Shop/Data_Output/weekly_clean_final.csv', parse_dates=['date'], index_col='date')
df_weekly = df_weekly.resample('W-MON').sum()
df_weekly = df_weekly[df_weekly.index <= '2026-02-09']

# Load Daily SKU Data (for disaggregation)
df_daily_sku = pd.read_csv('../01_data_ITA-Cake-Shop/Data_Output/Master_Clean_Feb2023_Feb2026.csv', parse_dates=['order date'])
df_daily_sku = df_daily_sku[df_daily_sku['order date'] <= '2026-02-09']

print("Data loaded for Simulation.")


Data loaded for Simulation.


## 1. 12-Week Expanding Window Forecast
Kita gunakan *expanding window* untuk menyimulasikan proses *batch* mingguan yang sebenarnya di lapangan. Model Champion dilatih dengan seluruh data hingga $T$, lalu memprediksi $T+1$.

In [2]:
json_path = 'Data_Output/champion_parameters.json'
with open(json_path, 'r') as f:
    champion_params_raw = json.load(f)

champion_params = {}
for cat, p in champion_params_raw.items():
    champion_params[cat] = {
        'model': p.get('seasonal', 'multiplicative'),
        'm': p.get('m', 26),
        'trend': p.get('trend', None),
        'alpha': p.get('alpha', 0.0),
        'beta': None if p.get('trend') is None else p.get('beta', 0.0),
        'gamma': p.get('gamma', 0.0),
        'init': p.get('init', 'estimated')
    }

sim_weeks = 12
forecast_results = {}
residuals_std = {}
weekly_sim_records = []

for cat in ['Cake', 'Japanese', 'Mille Crepes']:
    series = df_weekly[cat].copy()
    params = champion_params[cat]
    
    y_true = []
    y_pred = []
    
    # Calculate Residual STD for Safety Stock using the whole series up to 12 weeks ago
    train_full = series.iloc[:-sim_weeks]
    hw_res = ExponentialSmoothing(train_full.replace(0, 0.1), 
                                  trend=params['trend'], 
                                  seasonal=params['model'], 
                                  seasonal_periods=params['m'],
                                  initialization_method=params['init']).fit(
                                      smoothing_level=params['alpha'],
                                      smoothing_trend=params['beta'],
                                      smoothing_seasonal=params['gamma'],
                                      optimized=False
                                  )
    residuals_std[cat] = float(np.std(hw_res.resid.iloc[-26:], ddof=1))
    
    for i in range(sim_weeks, 0, -1):
        train = series.iloc[:-i]
        test = float(series.iloc[-i])
        date_val = series.index[-i]
        
        train_hw = train.copy()
        if params['model'] == 'mul':
            train_hw = train_hw.replace(0, 0.1)
            
        hw = ExponentialSmoothing(train_hw, 
                                  trend=params['trend'], 
                                  seasonal=params['model'], 
                                  seasonal_periods=params['m'],
                                  initialization_method=params['init']).fit(
                                      smoothing_level=params['alpha'],
                                      smoothing_trend=params['beta'],
                                      smoothing_seasonal=params['gamma'],
                                      optimized=False
                                  )
        pred = float(max(0.0, hw.forecast(1).values[0]))
        y_true.append(test)
        y_pred.append(pred)
        
        ss_weekly = 1.644854 * residuals_std[cat]
        tprod_weekly = np.round(pred + ss_weekly)
        weekly_sim_records.append({
            'Category': cat,
            'Date': date_val,
            'Actual': test,
            'Forecast': pred,
            'Safety_Stock': ss_weekly,
            'Target_Production': tprod_weekly,
            'Stockout': test > tprod_weekly
        })
        
    forecast_results[cat] = pd.DataFrame({
        'Date': series.index[-sim_weeks:],
        'Actual': y_true,
        'Forecast': y_pred
    }).set_index('Date')

df_weekly_records = pd.DataFrame(weekly_sim_records)

# WMAPE Evaluation
print("12-Week Operational WMAPE:")
for cat in ['Cake', 'Japanese', 'Mille Crepes']:
    res = forecast_results[cat]
    wmape = np.sum(np.abs(res['Actual'] - res['Forecast'])) / np.sum(res['Actual']) * 100
    print(f"{cat}: {wmape:.2f}%")

# Visualisasi Plot 6: operational_simulation_12w_tracking.png
COLOR_PALETTE = {
    'Cake': '#2b6cb0',
    'Japanese': '#d69e2e',
    'Mille Crepes': '#319795',
    'Actual': '#1a202c',
    'Forecast': '#2b6cb0',
    'Safety_Stock': '#ed8936',
    'Waste': '#e53e3e',
    'Fulfilled': '#38a169',
    'Target': '#805ad5'
}

fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
for ax, cat in zip(axes, ['Cake', 'Japanese', 'Mille Crepes']):
    sub_rec = df_weekly_records[df_weekly_records['Category'] == cat]
    dates = pd.to_datetime(sub_rec['Date'])
    
    ax.plot(dates, sub_rec['Actual'], color=COLOR_PALETTE['Actual'], lw=2.2, marker='s', ms=6, label=f'Aktual Mingguan ({cat})')
    ax.plot(dates, sub_rec['Forecast'], color=COLOR_PALETTE[cat], lw=2.0, marker='o', ms=5, linestyle='-', label='Forecast 1-Week-Ahead')
    ax.fill_between(dates, sub_rec['Forecast'], sub_rec['Target_Production'], color=COLOR_PALETTE['Safety_Stock'], alpha=0.3, label='Safety Stock Buffer (Z=1.65)')
    ax.step(dates, sub_rec['Target_Production'], color=COLOR_PALETTE['Safety_Stock'], lw=1.5, linestyle=':', where='mid', label='Target Produksi Mingguan')
    
    stockouts = sub_rec[sub_rec['Stockout']]
    if not stockouts.empty:
        ax.scatter(pd.to_datetime(stockouts['Date']), stockouts['Actual'], color=COLOR_PALETTE['Waste'], s=80, zorder=5, marker='X', label='Stockout Event')
        
    wmape = np.sum(np.abs(sub_rec['Actual'] - sub_rec['Forecast'])) / np.sum(sub_rec['Actual']) * 100
    ax.set_title(f'Tracking Kinerja Operasional: Kategori {cat} | 12-Week WMAPE: {wmape:.2f}%',
                 fontsize=11, fontweight='bold', loc='left')
    ax.set_ylabel('Volume (Pcs)')
    ax.legend(loc='upper right', frameon=True, fontsize=8, ncol=2)

axes[-1].set_xlabel('Tanggal Evaluasi Mingguan (W-MON)')
plt.tight_layout()
plt.savefig('Visualisasi_Gambar/operational_simulation_12w_tracking.png', dpi=300, bbox_inches='tight')
plt.show()

12-Week Operational WMAPE:


Cake: 24.77%
Japanese: 22.03%
Mille Crepes: 22.37%


## 2. Disagregasi Anti-Limbah (Day-of-Week & Rolling 12-Week SKU)
Tahap krusial ini menerapkan aturan *Rolling 12-Week SKU Share* dan *MTO Threshold* untuk mencegah produksi membabi-buta pada *SKU mati*.

In [3]:
def get_dow_share(df_daily, category, end_date, window_days=28):
    start_date = pd.to_datetime(end_date) - pd.Timedelta(days=window_days)
    mask = (df_daily['order date'] > start_date) & (df_daily['order date'] <= end_date) & (df_daily['item group'] == category)
    df_window = df_daily[mask]
    
    if df_window.empty or df_window['qty'].sum() == 0:
        return {i: 1/7 for i in range(7)}
        
    total_qty = df_window['qty'].sum()
    dow_qty = df_window.groupby(df_window['order date'].dt.dayofweek)['qty'].sum()
    
    dow_share = {i: dow_qty.get(i, 0) / total_qty for i in range(7)}
    return dow_share

def get_sku_share(df_daily, category, end_date, window_days=84): # 12 Weeks
    start_date = pd.to_datetime(end_date) - pd.Timedelta(days=window_days)
    mask = (df_daily['order date'] > start_date) & (df_daily['order date'] <= end_date) & (df_daily['item group'] == category)
    df_window = df_daily[mask]
    
    if df_window.empty or df_window['qty'].sum() == 0:
        # Fallback to uniform if no sales (unlikely for 12 weeks)
        unique_skus = df_daily[df_daily['item group'] == category]['item name'].unique()
        return {sku: 1/len(unique_skus) for sku in unique_skus}
        
    total_qty = df_window['qty'].sum()
    sku_qty = df_window.groupby('item name')['qty'].sum()
    
    return (sku_qty / total_qty).to_dict()

# Plot 7: operational_dow_adaptive_profiles.png
ALL_DAYS = pd.date_range(df_daily_sku['order date'].min(), df_daily_sku['order date'].max(), freq='D')
DOW_NAMES = ['Senin', 'Selasa', 'Rabu', 'Kamis', 'Jumat', 'Sabtu', 'Minggu']

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
for ax, cat in zip(axes, ['Cake', 'Japanese', 'Mille Crepes']):
    sub = df_daily_sku[df_daily_sku['item group'] == cat]
    daily_s = sub.groupby('order date')['qty'].sum().reindex(ALL_DAYS, fill_value=0)
    
    last_28d = daily_s.iloc[-28:]
    tot_28d = last_28d.sum()
    dow_28d_share = [last_28d[last_28d.index.dayofweek == d].sum() / tot_28d * 100 if tot_28d > 0 else 100/7 for d in range(7)]
    
    tot_all = daily_s.sum()
    dow_all_share = [daily_s[daily_s.index.dayofweek == d].sum() / tot_all * 100 if tot_all > 0 else 100/7 for d in range(7)]
    
    x = np.arange(7)
    width = 0.38
    ax.bar(x - width/2, dow_28d_share, width, color=COLOR_PALETTE[cat], alpha=0.85, label='4-Minggu Terakhir (Adaptif)')
    ax.bar(x + width/2, dow_all_share, width, color='#a0aec0', alpha=0.85, label='Historis 3-Tahun')
    ax.axhline(100/7, color='#e53e3e', linestyle=':', lw=1.2, label='Baseline Merata (14.29%)')
    
    ax.set_title(f'Profil DoW Share: {cat}', fontweight='bold', fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(DOW_NAMES, rotation=35, ha='right')
    ax.set_ylabel('Pangsa Permintaan (%)' if cat == 'Cake' else '')
    ax.legend(loc='upper left', frameon=True, fontsize=8)
    ax.grid(axis='y', alpha=0.6)

plt.tight_layout()
plt.savefig('Visualisasi_Gambar/operational_dow_adaptive_profiles.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot 8: operational_sku_pareto_distribution.png
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
sku_summary_list = []
for ax, cat in zip(axes, ['Cake', 'Japanese', 'Mille Crepes']):
    sub = df_daily_sku[df_daily_sku['item group'] == cat]
    sku_counts = sub.groupby('item name')['qty'].sum().sort_values(ascending=False)
    tot_qty = sku_counts.sum()
    
    sku_df = sku_counts.reset_index()
    sku_df.columns = ['SKU', 'Total_Qty']
    sku_df['Share (%)'] = (sku_df['Total_Qty'] / tot_qty) * 100
    sku_df['Cum_Share (%)'] = sku_df['Share (%)'].cumsum()
    sku_df['Pareto_Class'] = np.where(sku_df['Cum_Share (%)'] <= 80.0, 'A',
                               np.where(sku_df['Cum_Share (%)'] <= 95.0, 'B', 'C'))
    sku_df['Kategori'] = cat
    sku_summary_list.append(sku_df)
    
    top_n = min(10, len(sku_df))
    plot_df = sku_df.head(top_n)
    
    ax2 = ax.twinx()
    bars = ax.bar(range(top_n), plot_df['Total_Qty'], color=COLOR_PALETTE[cat], alpha=0.8, edgecolor='#1a202c', label='Volume Penjualan (Pcs)')
    line = ax2.plot(range(top_n), plot_df['Cum_Share (%)'], color='#e53e3e', lw=2.2, marker='o', ms=5, label='Kumulatif (%)')
    ax2.axhline(80, color='#d69e2e', linestyle='--', lw=1.2, label='Ambang Batas 80% (Kelas A)')
    
    ax.set_title(f'Pareto SKU: {cat} (Top {top_n} SKU)', fontweight='bold', fontsize=11)
    ax.set_xticks(range(top_n))
    short_labels = [s[:18] + '...' if len(s) > 18 else s for s in plot_df['SKU']]
    ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Total Penjualan (Pcs)')
    ax2.set_ylabel('Persentase Kumulatif (%)' if cat == 'Mille Crepes' else '')
    ax2.set_ylim(0, 105)
    ax2.grid(False)
    
    n_class_a = len(sku_df[sku_df['Pareto_Class'] == 'A'])
    ax.text(0.98, 0.45, f"Kelas A: {n_class_a} SKU\nTotal: {len(sku_df)} SKU",
            transform=ax.transAxes, ha='right', fontsize=8.5,
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.9, edgecolor='#cbd5e0'))

plt.tight_layout()
plt.savefig('Visualisasi_Gambar/operational_sku_pareto_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Kesimpulan Skripsi
Penerapan *MTO Threshold* ($<0.5$ dialihkan menjadi sistem *Make-to-Order* / *Pre-Order*), pembulatan aditif `np.round` (`Forecast_Base + Safety_Stock == Final_Production_Target`), serta simulasi inventori FIFO dengan masa simpan kue 6 hari (*shelf-life* 5–7 hari sesuai karakteristik produk *bakery* di Bab 1) berhasil menjaga keseimbangan optimal antara ketersediaan stok dan pengendalian limbah.

Dengan menggunakan deviasi standar residual 26 pekan terakhir dan proporsi disagregasi harian per SKU, sistem mampu mencapai *Service Level / Fill Rate* tinggi (**84,27%** pada Cake, **87,90%** pada Japanese, dan **96,60%** pada Mille Crepes; rata-rata gabungan **87,97%**) sekaligus menekan rasio produk kedaluwarsa (*Waste Ratio*) ke kisaran **4,36% hingga 16,45%** (rata-rata gabungan **12,93%**).


In [4]:
from scipy.stats import norm

# Ekspor Simulasi Operasional 12 Pekan
df_weekly_records.to_csv('Data_Output/operational_simulation_final.csv', index=False)
print('Saved operational_simulation_final.csv')

# Simulasi Disagregasi 12 Pekan (Manajemen Persediaan Service Level 95% & Masa Simpan 6 Hari FIFO)
disaggregated_plan = []
Z_SCORE = norm.ppf(0.95)  # Diturunkan secara formal dari scipy.stats untuk Service Level 95% (satu sisi)

daily_sales_dict = df_daily_sku.groupby(['order date', 'item group', 'item name'])['qty'].sum().to_dict()

for cat in ['Cake', 'Japanese', 'Mille Crepes']:
    for target_week_end in forecast_results[cat].index:
        weekly_forecast = forecast_results[cat].loc[target_week_end, 'Forecast']
        target_week_start = target_week_end - pd.Timedelta(days=6)
        dates = pd.date_range(target_week_start, target_week_end)
        
        dow_share = get_dow_share(df_daily_sku, cat, target_week_start)
        sku_share = get_sku_share(df_daily_sku, cat, target_week_start)
        std_resid = residuals_std[cat]
        
        for dt in dates:
            dow_val = dow_share.get(dt.dayofweek, 1.0/7.0)
            day_forecast = weekly_forecast * dow_val
            
            for sku, share in sku_share.items():
                if share <= 0:
                    continue
                
                sku_day_forecast = day_forecast * share
                # Disagregasi safety stock proporsional ke SKU & hari (konsisten dengan weekly_batch.py)
                safety_stock = (Z_SCORE * std_resid) * share * dow_val
                raw_target = sku_day_forecast + safety_stock
                
                # MTO Threshold (< 0.5) & Pembulatan Integer Aditif (Forecast_Base + Safety_Stock == Final_Production_Target)
                if raw_target < 0.5 or sku_day_forecast < 0.25:
                    fm_int = 0
                    ss_int = 0
                    final_target = 0
                else:
                    fm_int = int(np.round(sku_day_forecast))
                    ss_int = int(np.round(safety_stock))
                    if fm_int == 0 and raw_target >= 0.5:
                        fm_int = 1
                    final_target = fm_int + ss_int
                    
                actual_sales = int(daily_sales_dict.get((pd.Timestamp(dt.date()), cat, sku), 0))
                
                if final_target > 0 or actual_sales > 0:
                    disaggregated_plan.append({
                        'Date': dt.date(),
                        'Category': cat,
                        'SKU': sku,
                        'Forecast_Base': fm_int,
                        'Safety_Stock': ss_int,
                        'Final_Production_Target': final_target,
                        'Actual_Sales': actual_sales,
                        'Actual_Demand': actual_sales,
                        'Fulfilled_Qty': 0,
                        'Waste_Overproduction': 0,
                        'Waste_Qty': 0,
                        'Lost_Sales_Shortage': 0,
                        'Stockout_Qty': 0
                    })

plan_df = pd.DataFrame(disaggregated_plan)
plan_df['Date_dt'] = pd.to_datetime(plan_df['Date'])
plan_df.sort_values(['Category', 'SKU', 'Date_dt'], inplace=True)
plan_df.reset_index(drop=True, inplace=True)

# Simulasi Inventori FIFO dengan Masa Simpan (Shelf-Life) 6 Hari (Standar Operasional Chiller Toko)
SHELF_LIFE = 6
for (cat, sku), grp in plan_df.groupby(['Category', 'SKU']):
    batches = []  # List of [prod_date, remaining_qty]
    for idx, row in grp.iterrows():
        cur_dt = row['Date_dt']
        # 1. Kedaluwarsa batch yang umurnya >= SHELF_LIFE hari
        expired_today = 0
        active_batches = []
        for b_dt, b_qty in batches:
            if (cur_dt - b_dt).days >= SHELF_LIFE:
                expired_today += b_qty
            else:
                active_batches.append([b_dt, b_qty])
        batches = active_batches
        
        # 2. Tambah produksi baru hari ini
        prod_today = int(row['Final_Production_Target'])
        if prod_today > 0:
            batches.append([cur_dt, prod_today])
            
        # 3. Penuhi permintaan aktual secara FIFO
        demand_today = int(row['Actual_Sales'])
        rem_demand = demand_today
        for b in batches:
            if rem_demand == 0:
                break
            used = min(b[1], rem_demand)
            b[1] -= used
            rem_demand -= used
        batches = [b for b in batches if b[1] > 0]
        
        fulfilled_today = demand_today - rem_demand
        shortage_today = rem_demand
        
        plan_df.loc[idx, 'Fulfilled_Qty'] = int(fulfilled_today)
        plan_df.loc[idx, 'Waste_Overproduction'] = int(expired_today)
        plan_df.loc[idx, 'Waste_Qty'] = int(expired_today)
        plan_df.loc[idx, 'Lost_Sales_Shortage'] = int(shortage_today)
        plan_df.loc[idx, 'Stockout_Qty'] = int(shortage_today)

plan_df.drop(columns=['Date_dt'], inplace=True)
display(plan_df.head())

plan_df.to_csv('Data_Output/production_targets_daily_final.csv', index=False)
print('Saved production_targets_daily_final.csv')

# Tabel Ringkasan Limbah (Waste Analysis)
waste_summary = plan_df.groupby('Category')[['Forecast_Base', 'Safety_Stock', 'Final_Production_Target', 'Actual_Sales', 'Fulfilled_Qty', 'Waste_Overproduction', 'Lost_Sales_Shortage']].sum().reset_index()
waste_summary['Fill_Rate_%'] = (waste_summary['Fulfilled_Qty'] / waste_summary['Actual_Sales'] * 100).round(2)
waste_summary['Waste_Ratio_%'] = (waste_summary['Waste_Overproduction'] / waste_summary['Final_Production_Target'] * 100).fillna(0).round(2)
display(waste_summary)

waste_summary.to_csv('Data_Output/waste_analysis_final.csv', index=False)
print('Saved waste_analysis_final.csv')

# Plot 9: operational_daily_production_vs_demand.png
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
for ax, cat in zip(axes, ['Cake', 'Japanese', 'Mille Crepes']):
    daily_cat_agg = plan_df[plan_df['Category'] == cat].groupby('Date')[['Final_Production_Target', 'Actual_Demand']].sum().reset_index()
    daily_cat_agg['Date'] = pd.to_datetime(daily_cat_agg['Date'])
    
    ax.plot(daily_cat_agg['Date'], daily_cat_agg['Actual_Demand'], color=COLOR_PALETTE['Actual'], lw=1.8, marker='o', ms=3, label='Permintaan Harian Aktual')
    ax.plot(daily_cat_agg['Date'], daily_cat_agg['Final_Production_Target'], color=COLOR_PALETTE['Target'], lw=1.8, linestyle='-', label='Target Produksi Harian (Forecast + SS)')
    ax.fill_between(daily_cat_agg['Date'], daily_cat_agg['Actual_Demand'], daily_cat_agg['Final_Production_Target'],
                    where=daily_cat_agg['Final_Production_Target'] >= daily_cat_agg['Actual_Demand'],
                    color=COLOR_PALETTE['Safety_Stock'], alpha=0.25, label='Buffer Safety Stock / Terpenuhi')
    ax.fill_between(daily_cat_agg['Date'], daily_cat_agg['Final_Production_Target'], daily_cat_agg['Actual_Demand'],
                    where=daily_cat_agg['Actual_Demand'] > daily_cat_agg['Final_Production_Target'],
                    color=COLOR_PALETTE['Waste'], alpha=0.4, label='Kejadian Stockout Harian')
    
    ax.set_title(f'Dinamika Target Produksi vs Permintaan Riil Harian: {cat} (84 Hari Evaluasi)', fontweight='bold', fontsize=11, loc='left')
    ax.set_ylabel('Unit (Pcs)')
    ax.legend(loc='upper right', frameon=True, fontsize=8, ncol=2)

axes[-1].set_xlabel('Tanggal Operasional Harian')
plt.tight_layout()
plt.savefig('Visualisasi_Gambar/operational_daily_production_vs_demand.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot 10: operational_waste_and_fill_rate.png
waste_cat_summary = plan_df.groupby('Category').agg({
    'Final_Production_Target': 'sum',
    'Actual_Demand': 'sum',
    'Fulfilled_Qty': 'sum',
    'Waste_Qty': 'sum',
    'Stockout_Qty': 'sum'
}).reindex(['Cake', 'Japanese', 'Mille Crepes']).reset_index()

waste_cat_summary['Fill Rate (%)'] = (waste_cat_summary['Fulfilled_Qty'] / waste_cat_summary['Actual_Demand']) * 100
waste_cat_summary['Waste Rate (%)'] = (waste_cat_summary['Waste_Qty'] / waste_cat_summary['Final_Production_Target']) * 100
waste_cat_summary['Service Level (%)'] = np.minimum(100.0, waste_cat_summary['Fill Rate (%)'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14.5, 5.5))
x_cats = np.arange(3)
width = 0.35

b1 = ax1.bar(x_cats - width/2, waste_cat_summary['Service Level (%)'], width, color=COLOR_PALETTE['Fulfilled'], label='Customer Service Level / Fill Rate (%)')
b2 = ax1.bar(x_cats + width/2, waste_cat_summary['Waste Rate (%)'], width, color=COLOR_PALETTE['Waste'], alpha=0.85, label='Expired Waste Ratio (Shelf-Life 6 Hari) (%)')

ax1.axhline(95, color='#2f855a', linestyle='--', lw=1.2, label='Target Service Level 95%')
ax1.set_title('Trade-Off: Service Level (Fill Rate) vs Rasio Waste Kedaluwarsa', fontweight='bold', fontsize=11)
ax1.set_xticks(x_cats)
ax1.set_xticklabels(['Cake', 'Japanese', 'Mille Crepes'], fontweight='bold')
ax1.set_ylabel('Persentase (%)')
ax1.set_ylim(0, 122)
ax1.legend(loc='upper right', frameon=True, fontsize=8.5)

for bar in b1:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, h + 1.8, f'{h:.1f}%', ha='center', va='bottom', fontsize=8.5, fontweight='bold', color='#22543d')
for bar in b2:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, h + 1.8, f'{h:.1f}%', ha='center', va='bottom', fontsize=8.5, fontweight='bold', color='#742a2a')

sold_carryover_qty = waste_cat_summary['Final_Production_Target'] - waste_cat_summary['Waste_Qty']
p1 = ax2.bar(['Cake', 'Japanese', 'Mille Crepes'], sold_carryover_qty, color=COLOR_PALETTE['Fulfilled'], width=0.45, label='Volume Terpakai / Carry-Over Chiller (Pcs)')
p2 = ax2.bar(['Cake', 'Japanese', 'Mille Crepes'], waste_cat_summary['Waste_Qty'], bottom=sold_carryover_qty, color=COLOR_PALETTE['Waste'], alpha=0.85, width=0.45, label='Volume Kedaluwarsa / Waste > 6 Hari (Pcs)')

ax2.set_title('Komposisi Target Produksi 12 Pekan: Terpakai vs Waste', fontweight='bold', fontsize=11)
ax2.set_ylabel('Total Unit Produksi (Pcs)')
ax2.set_ylim(0, waste_cat_summary['Final_Production_Target'].max() * 1.28)
ax2.legend(loc='upper right', frameon=True, fontsize=8.5)

for i, cat in enumerate(['Cake', 'Japanese', 'Mille Crepes']):
    tot = int(waste_cat_summary.loc[i, 'Final_Production_Target'])
    ful = int(waste_cat_summary.loc[i, 'Fulfilled_Qty'])
    wst = int(waste_cat_summary.loc[i, 'Waste_Qty'])
    ax2.text(i, tot + 45, f'Target: {tot:,} pcs\n(Terpenuhi: {ful:,} | Waste: {wst:,})', ha='center', va='bottom', fontsize=8.3, fontweight='bold')

plt.tight_layout()
plt.savefig('Visualisasi_Gambar/operational_waste_and_fill_rate.png', dpi=300, bbox_inches='tight')
plt.show()

Saved operational_simulation_final.csv
Saved production_targets_daily_final.csv
    Category  Forecast_Base  Safety_Stock  Final_Production_Target  Actual_Sales  Fulfilled_Qty  Waste_Overproduction  Lost_Sales_Shortage  Fill_Rate_%  Waste_Ratio_%
        Cake           1666           404                     2070          1564           1318                   317                  246        84.27          15.31
    Japanese            963           253                     1216           868            763                   200                  105        87.90          16.45
Mille Crepes            666           411                     1077           677            654                    47                   23        96.60           4.36
Saved waste_analysis_final.csv
